In [7]:
!pip install ucimlrepo

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\patyq\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import os 
# fetch dataset 
breast_cancer_wisconsin_original = fetch_ucirepo(id=15) 
  
# data (as pandas dataframes) 
X = breast_cancer_wisconsin_original.data.features 
y = breast_cancer_wisconsin_original.data.targets 
df = pd.concat([X, y], axis=1)
df.to_csv('../data/breast_cancer_original.csv', index=False)

### 1. Comparaciones (Class 2 vs Class 4)
-  En tumores benignos (Class = 2): Las variables como Uniformity_of_cell_size y Uniformity_of_cell_shape se mantienen en valores muy bajos (1, 2 o 3).
- En tumores malignos (Class = 4): Esas mismas variables se disparan a valores altos (8, 9 o 10).

Al hacer la tabla cruzada o comparar promedios, salta a la vista de inmediato que la uniformidad celular es una variable con un alto poder discriminatorio.

### 2. Cómo se verían las 2 Representaciones con tus datos

Justificación del Limpiado Previo (Común para ambas propuestas)
- Exclusión de Sample_code_number: Se descarta por ser un identificador administrativo de paciente que no aporta valor informativo ni biológico, previniendo el sobreajuste (overfitting) por memorización de patrones irrelevantes [7, 8].
- Imputación por mediana en Bare_nuclei: Permite conservar la totalidad del conjunto de datos original sin descartar observaciones, empleando la mediana como un estimador estadístico robusto ante sesgos en variables discretas [7, 8].

In [9]:
# ==============================================================================
# 1. CARGA DE DATOS
# ==============================================================================
df_ori = pd.read_csv('../data/breast_cancer_original.csv')

print(f"Dimensiones iniciales: {df_ori.shape}")

# ==============================================================================
# 2. LIMPIEZA DE IDENTIFICADORES Y DUPLICADOS
# ==============================================================================
# Eliminar la columna de ID
if 'Sample_code_number' in df_ori.columns:
    df_ori = df_ori.drop(columns=['Sample_code_number'])

# Eliminar filas duplicadas (si existen)
duplicados = df_ori.duplicated().sum()
df_ori = df_ori.drop_duplicates()
print(f"Filas duplicadas eliminadas: {duplicados}")

# ==============================================================================
# PASO PREVIO: Imputación de Valores Faltantes
# ==============================================================================

# Lista de las 9 características clínicas (excluyendo el ID: Sample_code_number)
feature_cols = [
    'Clump_thickness', 'Uniformity_of_cell_size', 'Uniformity_of_cell_shape',
    'Marginal_adhesion', 'Single_epithelial_cell_size', 'Bare_nuclei',
    'Bland_chromatin', 'Normal_nucleoli', 'Mitoses'
]

# MANEJO DE VALORES FALTANTES (Bare_nuclei)
# Reemplazar el carácter '?' por NaN
df_ori['Bare_nuclei'] = pd.to_numeric(df_ori['Bare_nuclei'].replace('?', np.nan))

# Imputamos los datos ausentes de 'Bare_nuclei' usando la mediana
imputer = SimpleImputer(strategy='median')
feature_cols = [col for col in df_ori.columns if col != 'Class']
df_ori[feature_cols] = imputer.fit_transform(df_ori[feature_cols])

print(f"Valores nulos restantes en el dataset: {df_ori.isnull().sum().sum()}")

# 4. RECODIFICACIÓN DE LA VARIABLE OBJETIVO (Class)
# Convertir 2 (Benigno) -> 0,  4 (Maligno) -> 1
df_ori['Class'] = df_ori['Class'].map({2: 0, 4: 1})

# Guardar el conjunto de datos limpio y listo para modelar
os.makedirs('../data', exist_ok=True)
df_ori.to_csv('../data/breast_cancer_limpio.csv', index=False)
print("¡Limpieza y preprocesamiento completados con éxito! Archivo 'breast_cancer_limpio.csv' guardado.")

Dimensiones iniciales: (699, 10)
Filas duplicadas eliminadas: 236
Valores nulos restantes en el dataset: 0
¡Limpieza y preprocesamiento completados con éxito! Archivo 'breast_cancer_limpio.csv' guardado.


#### Representación 1: Numérica Estandarizada (Z-score)
- Objetivo: Mantener la variable como continua pero llevarla a media 0 y desviación estándar 
- 1.Resultado numérico:
    - Los valores originales 1 (como en las filas 0, 2, 4) se convierten en valores negativos (ejemplo: -0.58).
    - El valor 10 de la fila 5 se convierte en un valor positivo grande (ejemplo: +2.41).
- Uso: Ideal si vas a usar Naive Bayes, Regresión Logística o algoritmos basados en distancias.


In [10]:
# ==============================================================================
# REPRESENTACIÓN 1: Numérica Estandarizada (Z-score)
# Ideal para: Naive Bayes, Regresión Logística, K-NN, Redes Neuronales
# ==============================================================================

df_1 = pd.read_csv('../data/breast_cancer_limpio.csv')
scaler = StandardScaler()
X_zscore = scaler.fit_transform(df_1[feature_cols])

# Crear DataFrame de la Representación 1
df_rep1 = pd.DataFrame(X_zscore, columns=[f"{col}_zscore" for col in feature_cols])
df_rep1['Class'] = df_1['Class'].values  # Conservar la variable objetivo

# Guardar en archivo CSV
df_rep1.to_csv('../data/representacion_1_zscore.csv', index=False)
print(" Representación 1 (Z-Score) generada con éxito.")


 Representación 1 (Z-Score) generada con éxito.


Justificación de la Representación 1: Numérica Estandarizada (Z-score)
- Técnicas aplicadas: Imputación por mediana de la variable Bare_nuclei y estandarización Z-score en las 9 características clínicas (media 0, desviación estándar 1) [7, 8].
- Fundamento en Ingeniería de Características: Mantiene la naturaleza continua y cuantitativa original de las mediciones histológicas sin distorsionar la distribución ni la distancia relativa entre las observaciones [7, 8]. Al igualar la media y la varianza de todas las variables, se evita que aquellos atributos con mayor dispersión dominen injustificadamente sobre los demás durante el aprendizaje [7, 8].
- Idoneidad Algorítmica: Es la representación recomendada para clasificadores probabilísticos como Naive Bayes (al trabajar sobre distribuciones numéricas continuas homogeneizadas) y para modelos basados en distancias como K-Nearest Neighbors (KNN), Máquinas de Vectores de Soporte (SVM) o Regresión Logística [1, 3, 5].

#### Representación 2: Discretización por Rangos de Severidad (Binning)
- Objetivo: Convertir la escala del 1 al 10 en categorías clínicas interpretables.
- Criterio de agrupación:
    - 1 a 3 $\rightarrow$ 0 (Normal / Bajo)
    - 4 a 6 $\rightarrow$ 1 (Moderado)
    - 7 a 10 $\rightarrow$ 2 (Alto / Atípico)
- Resultado en la muestra:
    - Fila 0 (Benigno, valor 1) $\rightarrow$ 0 (Normal)
    - Fila 1 (Benigno, valor 4) $\rightarrow$ 1 (Moderado)
    - Fila 5 (Maligno, valor 10) $\rightarrow$ 2 (Alto / Atípico)
- Uso: Ideal para generar reglas claras de corte en Árboles de Decisión.

In [11]:
# ==============================================================================
# REPRESENTACIÓN 2: Discretización por Rangos Clínicos (Binning)
# Ideal para: Árboles de Decisión, Reglas de Asociación
# ==============================================================================
df_rep2 = pd.DataFrame()
df_2 = pd.read_csv('../data/breast_cancer_limpio.csv')

for col in feature_cols:
    # Convertir '?' a NaN y forzar tipo numérico
    df_2[col] = pd.to_numeric(df_2[col].replace('?', np.nan))
    # Rellenar cualquier NaN previo con la mediana
    df_2[col] = df_2[col].fillna(df_2[col].median())

# Definir los rangos de severidad clínica:
# 1-3 -> 0 (Bajo / Normal)
# 4-6 -> 1 (Moderado)
# 7-10 -> 2 (Alto / Atípico)
bins = [0, 3, 6, 10]
labels = [0, 1, 2]

# 4. Discretizar usando la opción de enteros nulos 'Int64' (con "I" mayúscula)
for col in feature_cols:
    df_rep2[f"{col}_bin"] = pd.cut(
        df_2[col], 
        bins=bins, 
        labels=labels
    ).astype('Int64')

df_rep2['Class'] = df_2['Class'].values
df_rep2.to_csv('../data/representacion_2_binning.csv', index=False)
print("Representación 2 (Discretizada) generada con éxito.")

Representación 2 (Discretizada) generada con éxito.


El binning (discretización) agrupa valores numéricos en intervalos o categorías. En tu proyecto sirve para:
- Reducir ruido: Suaviza variaciones menores de medición (ej. agrupar valores 2 y 3 en la categoría Bajo).
- Mejorar la interpretabilidad: Traduce escalas del 1 al 10 a rangos clínicos intuitivos (0: Bajo, 1: Moderado, 2: Alto).
- Optimizar Árboles de Decisión: Facilita divisiones (splits) más simples y directas, evitando el sobreajuste (overfitting).
- Capturar umbrales no lineales: Modela saltos bruscos de riesgo médico que no crecen de forma lineal.
- Simplificar características: Reduce la complejidad del conjunto de datos transformando atributos continuos en discretos.

Justificación de la Representación 2: Discretización por Rangos Clínicos (Binning)
- Técnicas aplicadas: Imputación por mediana y agrupamiento en 3 categorías ordinales de severidad (0: Bajo/Normal, 1: Moderado, 2: Alto/Atípico) [7, 8].
- Fundamento en Ingeniería de Características: Emula el proceso de diagnóstico médico al traducir escalas cuantitativas (del 1 al 10) en niveles cualitativos de atipia celular [7, 8]. Esta reducción de resolución actúa como un filtro de ruido que absorbe pequeñas variaciones o errores menores de medición en el laboratorio [7, 8].
- Idoneidad Algorítmica: Es la representación óptima para modelos basados en reglas e interacciones como los Árboles de Decisión [1, 2, 5]. Al evaluar divisiones (splits) sobre intervalos discretos u ordinales, el árbol construye reglas de decisión sencillas, visuales e interpretables en términos clínicos (por ejemplo: Si la uniformidad del tamaño celular es Alta, la probabilidad de malignidad se incrementa) [1, 2, 5].

------------------------------------------------------------

1. ¿Cómo hacer una Tabla Cruzada (pd.crosstab)?

In [12]:
# Supongamos que usas la versión discretizada ('Uniformity_of_cell_size_bin')
# 0: Bajo, 1: Moderado, 2: Alto

tabla_cruzada = pd.crosstab(
    index=df_rep2['Uniformity_of_cell_size_bin'], 
    columns=df_rep2['Class'],
    margins=True,            # Agrega totales de filas y columnas
    normalize='columns'      # Muestra los resultados en porcentaje por columna (clase)
)

print(tabla_cruzada)

Class                               0         1       All
Uniformity_of_cell_size_bin                              
0                            0.937778  0.151261  0.533477
1                            0.048889  0.361345  0.209503
2                            0.013333  0.487395  0.257019


El porcentaje de pacientes con tumor benigno (2) tienen valores bajos frente a qué porcentaje de pacientes con tumor maligno (4) tienen valores altos.

2. ¿Cómo Comparar Promedios (df.groupby().mean())?

Para variables cuantitativas numéricas, la forma más rápida de comparar los valores promedio entre grupos es agrupando los datos según la clase del tumor mediante la técnica de agregación
:

In [13]:
# Comparar el promedio de todas las variables numéricas separadas por tipo de tumor
promedios_por_clase = df.groupby('Class')[feature_cols].mean()

print(promedios_por_clase)

       Clump_thickness  Uniformity_of_cell_size  Uniformity_of_cell_shape  \
Class                                                                       
2             2.956332                 1.325328                  1.443231   
4             7.195021                 6.572614                  6.560166   

       Marginal_adhesion  Single_epithelial_cell_size  Bare_nuclei  \
Class                                                                
2               1.364629                     2.120087     1.346847   
4               5.547718                     5.298755     7.627615   

       Bland_chromatin  Normal_nucleoli   Mitoses  
Class                                              
2             2.100437         1.290393  1.063319  
4             5.979253         5.863071  2.589212  


Evaluar la diferencia entre grupos para una sola variable (por ejemplo, Clump_thickness):

In [14]:
# Promedio de grosor del tumor según la clase
promedio_grosor = df.groupby('Class')['Clump_thickness'].agg(['mean', 'std', 'median'])

print(promedio_grosor)

           mean       std  median
Class                            
2      2.956332  1.674318     3.0
4      7.195021  2.428849     8.0


#### Resumen
Comparación de Promedios (groupby): Permite argumentar cuantitativamente qué características tienen una diferencia de medias más marcada entre tumores benignos y malignos.
Tabla Cruzada (crosstab): Ayuda a mostrar visualmente cómo se concentran las categorías discretas en cada clase de diagnóstico.